In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np 

In [8]:
device = torch.device("mps")

In [9]:
num_epochs = 4
batch_size = 4
learning_rate = 0.001

In [10]:
# dataset has pilimage images of range [0,1]. We need to transform them to Tensors of normalized range [-1, 1]
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


In [11]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [13]:
class ConvNet(nn.Module): 
    def __init__(self): 
        super(ConvNet, self).__init__()

        # input channel size = 3, output channel size = 6, kernel 5 x 5 
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2) # kernel = 2, stride = 2 (jump 2px to the right)
        self.conv2 = nn.Conv2d(6, 16, 5) 
        
        # fully connected layers
        # 16 channels, 5x5 image size after all conv/pool
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x) # no optimizable parameters (weights) here, func is const
        x = self.pool(x) # no optmizable parameters here, just max of the 2x2 kernel

        x = self.conv2(x)
        x = F.relu(x) 
        x = self.pool(x)

        # -1 = number of batches (4)
        x = x.view(-1, 16 * 5 * 5)

        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x) # no relu here, we want the raw outputs in all 10 neurons

        # softmax included in the loss
        return x


model = ConvNet().to(device)

criterion = nn.CrossEntropyLoss() # multi-class classification problem, softmax included
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # gradient descent

In [14]:
n_total_steps = len(train_loader) 

for epoch in range(num_epochs): 
    for i, (images, labels) in enumerate(train_loader): 
        # original shape: [4, 3, 32, 32]
        # input layer: 3 input channels, 6 output channels, 5 kernel size
        images = images.to(device) 
        labels = labels.to(device)

        # forward pass
        outputs = model(images) 
        loss = criterion(outputs, labels) 

        # backward and optimize 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 2000 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{n_total_steps}], Loss: {loss.item():.4f}')

Epoch [1/4], Step [2000/12500], Loss: 2.2801
Epoch [1/4], Step [4000/12500], Loss: 2.2962
Epoch [1/4], Step [6000/12500], Loss: 2.1963
Epoch [1/4], Step [8000/12500], Loss: 2.0905
Epoch [1/4], Step [10000/12500], Loss: 2.1428
Epoch [1/4], Step [12000/12500], Loss: 1.9145
Epoch [2/4], Step [2000/12500], Loss: 1.3887
Epoch [2/4], Step [4000/12500], Loss: 1.5870
Epoch [2/4], Step [6000/12500], Loss: 1.7999
Epoch [2/4], Step [8000/12500], Loss: 2.2657
Epoch [2/4], Step [10000/12500], Loss: 2.7727
Epoch [2/4], Step [12000/12500], Loss: 1.5844
Epoch [3/4], Step [2000/12500], Loss: 1.8368
Epoch [3/4], Step [4000/12500], Loss: 1.9496
Epoch [3/4], Step [6000/12500], Loss: 2.0878
Epoch [3/4], Step [8000/12500], Loss: 1.8064
Epoch [3/4], Step [10000/12500], Loss: 0.6882
Epoch [3/4], Step [12000/12500], Loss: 1.4066
Epoch [4/4], Step [2000/12500], Loss: 1.0929
Epoch [4/4], Step [4000/12500], Loss: 1.7098
Epoch [4/4], Step [6000/12500], Loss: 1.5155
Epoch [4/4], Step [8000/12500], Loss: 1.8678
Epoc

In [15]:
with torch.no_grad(): 
    n_correct = 0
    n_samples = 0
    n_class_correct = [0 for i in range(10)]
    n_class_samples = [0 for i in range(10)]

    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)

        # max returns (value, index)
        _, predicted = torch.max(outputs, 1)
        n_samples += labels.size(0)
        n_correct += (predicted == labels).sum().item()

        for i in range(batch_size): 
            label = labels[i]
            pred = predicted[i]
            if (label == pred): 
                n_class_correct[label] += 1
            n_class_samples[label] += 1

    # accuracy of the network
    acc = 100.0 * n_correct / n_samples
    print(f'Accuracy of the network: {acc} %')

    # accuracy of each class
    for i in range(10):
        acc = 100.0 * n_class_correct[i] / n_class_samples[i]
        print(f'Accuracy of {classes[i]}: {acc} %')

Accuracy of the network: 47.81 %
Accuracy of plane: 58.1 %
Accuracy of car: 63.4 %
Accuracy of bird: 26.0 %
Accuracy of cat: 33.1 %
Accuracy of deer: 32.9 %
Accuracy of dog: 29.7 %
Accuracy of frog: 71.6 %
Accuracy of horse: 57.7 %
Accuracy of ship: 49.9 %
Accuracy of truck: 55.7 %
